# Transcribe Public-Meeting Audio with WhisperX + Speaker Diarization

This notebook is deliberately limited to the **audio → transcript** stage of the project. It does **not** perform LDA, topic modeling, sentiment analysis, or downstream NLP.

The pipeline is explicit and linear:

**MP3 → WhisperX ASR → forced alignment → speaker diarization → speaker assignment → exports → QA**

## Why this version exists

For the hometown data-center case study, speaker identity and speaker boundaries matter. This rewrite makes diarization a first-class stage and preserves intermediate outputs so that every transformation is inspectable.

## Outputs for each MP3

- `*_asr.json` — raw WhisperX transcription before forced alignment.
- `*_aligned.json` — aligned transcript before speaker labels are attached.
- `*_diarization.csv` — raw diarization intervals returned by the speaker model.
- `*_whisperx.json` — final aligned + diarized WhisperX result.
- `*_transcript.txt` — plain text only, with no speaker-label tokens.
- `*_speakers.txt` — timestamped transcript with anonymous speaker IDs.
- `*_segments.csv` — one row per aligned transcript segment.
- `*_words.csv` — one row per aligned word, including speaker assignment when available.
- `*_run_metadata.json` — model settings, source-file hash, and output counts.
- `run_manifest.csv` — one-row summary for every MP3 processed.

### Important: what diarization does and does not do

Diarization estimates **who spoke when** and produces anonymous labels such as `SPEAKER_00`. It does not know that a voice belongs to a particular resident, official, attorney, or developer representative.

Treat speaker labels as **recording-specific identifiers**. Do not assume that `SPEAKER_03` in hearing 1 is the same human being as `SPEAKER_03` in hearing 2. A later annotation step can map each `(hearing, speaker ID)` pair to a real name and role.

## 1. Install the Python dependencies

The notebook pins WhisperX to the same stable release used in the earlier transcription notebook so the environment is explicit and reproducible.

`ffmpeg` is an operating-system dependency and is **not installed automatically** here. The next cell checks for it.

In [ ]:
print(COMPUTE_TYPE)

In [ ]:
%pip install -U "whisperx==3.8.6" pandas anthropic

## 2. Check `ffmpeg`

WhisperX needs `ffmpeg` to read MP3 audio. On macOS with Homebrew, the usual installation command is `brew install ffmpeg`.

This notebook only checks whether `ffmpeg` is visible; it does not make operating-system changes.

In [1]:
import shutil

ffmpeg_path = shutil.which("ffmpeg")
if ffmpeg_path is None:
    raise RuntimeError(
        "ffmpeg was not found on PATH. Install ffmpeg at the operating-system "
        "level, restart the Jupyter kernel if necessary, and rerun this cell."
    )
print("ffmpeg:", ffmpeg_path)

ffmpeg: /opt/homebrew/bin/ffmpeg


## 3. Imports and environment information

Record the runtime actually used for transcription. These versions are part of the provenance trail.

In [2]:
from pathlib import Path
from datetime import datetime, timezone
import gc
import hashlib
import json
import os
import platform
import sys

import pandas as pd
import torch
import whisperx
from whisperx.diarize import DiarizationPipeline

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("WhisperX:", getattr(whisperx, "__version__", "version attribute unavailable"))

Python: 3.10.18
Platform: macOS-26.5.1-arm64-arm-64bit
PyTorch: 2.8.0
WhisperX: version attribute unavailable


objc[29260]: Class AVFFrameReceiver is implemented in both /opt/homebrew/lib/python3.10/site-packages/av/.dylibs/libavdevice.62.3.101.dylib (0x11420c3a8) and /opt/homebrew/Cellar/ffmpeg@7/7.1.5_2/lib/libavdevice.61.3.100.dylib (0x12aff0328). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[29260]: Class AVFAudioReceiver is implemented in both /opt/homebrew/lib/python3.10/site-packages/av/.dylibs/libavdevice.62.3.101.dylib (0x11420c3f8) and /opt/homebrew/Cellar/ffmpeg@7/7.1.5_2/lib/libavdevice.61.3.100.dylib (0x12aff0378). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.


## 4. User configuration

This is the main cell to edit.

### Audio files
List the two raw MP3s explicitly. Missing files cause a visible error rather than being silently ignored.

### ASR model
`large-v3` is selected because transcript accuracy matters for close reading and quotation checking. If you intentionally choose another model, the choice is recorded in each run's metadata.

### Diarization
Diarization is enabled by default. Leave `MIN_SPEAKERS` and `MAX_SPEAKERS` as `None` unless you have a defensible reason to constrain the number of voices.

### Hugging Face token
Do **not** paste the token into the notebook. Store it in the environment variable `HF_TOKEN`.

In [3]:
# -------------------------
# AUDIO INPUTS
# -------------------------
AUDIO_FILES = [
    Path("CyrusOne/Cyrusone_Hearing_12032025.mp3"),
    Path("CyrusOne/COB_20260323_Imported.mp3"),
]

# -------------------------
# TRANSCRIPTION
# -------------------------
LANGUAGE = "en"
MODEL_NAME = "large-v3"
GPU_BATCH_SIZE = 16
CPU_BATCH_SIZE = 4

# -------------------------
# SPEAKER DIARIZATION
# -------------------------
ENABLE_DIARIZATION = True
DIARIZATION_MODEL = "pyannote/speaker-diarization-community-1"
MIN_SPEAKERS = None
MAX_SPEAKERS = None

# -------------------------
# OUTPUT / RESTART BEHAVIOR
# -------------------------
OUTPUT_DIR = Path("whisperx_outputs_diarized")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OVERWRITE = False

# -------------------------
# HARDWARE
# -------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_TYPE = "float16" if DEVICE == "cuda" else "int8"
BATCH_SIZE = GPU_BATCH_SIZE if DEVICE == "cuda" else CPU_BATCH_SIZE

print("Audio files:")
for p in AUDIO_FILES:
    print("  ", p)
print()
print("Device:             ", DEVICE)
print("Compute type:       ", COMPUTE_TYPE)
print("ASR model:          ", MODEL_NAME)
print("Batch size:         ", BATCH_SIZE)
print("Language:           ", LANGUAGE)
print("Diarization:        ", ENABLE_DIARIZATION)
print("Diarization model:  ", DIARIZATION_MODEL if ENABLE_DIARIZATION else "disabled")
print("Speaker constraints:", MIN_SPEAKERS, MAX_SPEAKERS)
print("Output directory:   ", OUTPUT_DIR.resolve())
print("Overwrite:          ", OVERWRITE)

Audio files:
   CyrusOne/Cyrusone_Hearing_12032025.mp3
   CyrusOne/COB_20260323_Imported.mp3

Device:              cpu
Compute type:        int8
ASR model:           large-v3
Batch size:          4
Language:            en
Diarization:         True
Diarization model:   pyannote/speaker-diarization-community-1
Speaker constraints: None None
Output directory:    /Users/yams/Library/CloudStorage/Dropbox/AEI_magazine_datacenters/whisperx_outputs_diarized
Overwrite:           False


## 5. Validate configuration before loading large models

This catches two common failures early: missing MP3 paths and diarization enabled without a Hugging Face token. The token is read from the environment and is never written to output metadata.

In [4]:
from getpass import getpass

missing_audio = [str(p) for p in AUDIO_FILES if not p.exists()]
if missing_audio:
    raise FileNotFoundError(
        "The following configured audio files do not exist:\n\n"
        + "\n".join(f"- {p}" for p in missing_audio)
        + "\n\nEdit AUDIO_FILES so the paths exactly match the MP3s on disk."
    )

HF_TOKEN = None
if ENABLE_DIARIZATION:
    HF_TOKEN = getpass("Enter your HF_TOKEN: ")
    if not HF_TOKEN:
        raise EnvironmentError(
            "ENABLE_DIARIZATION=True, but HF_TOKEN was not found. Create a "
            "Hugging Face read token, accept the diarization model access "
            "conditions, set HF_TOKEN in your environment, restart the kernel "
            "if necessary, and rerun."
        )

print("Configuration validation passed.")

Enter your HF_TOKEN: ········
Configuration validation passed.


## 6. Helper functions

These functions do not call any models. They format timestamps, hash the source MP3, serialize WhisperX results, and build plain-text, segment-level, and word-level exports.

The raw, aligned, and final transcripts remain separate throughout the pipeline.

In [5]:
def utc_now_iso():
    return datetime.now(timezone.utc).isoformat()


def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


def format_timestamp(seconds):
    if seconds is None:
        return "??:??:??.???"
    milliseconds = int(round(float(seconds) * 1000))
    hours, remainder = divmod(milliseconds, 3_600_000)
    minutes, remainder = divmod(remainder, 60_000)
    secs, ms = divmod(remainder, 1_000)
    return f"{hours:02d}:{minutes:02d}:{secs:02d}.{ms:03d}"


def json_default(obj):
    if hasattr(obj, "item"):
        return obj.item()
    if hasattr(obj, "tolist"):
        return obj.tolist()
    return str(obj)


def write_json(obj, path):
    with Path(path).open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=json_default)


def make_plain_transcript(result):
    pieces = []
    for segment in result.get("segments", []):
        text = str(segment.get("text", "")).strip()
        if text:
            pieces.append(text)
    return "\n".join(pieces)


def make_speaker_transcript(result):
    lines = []
    for segment in result.get("segments", []):
        text = str(segment.get("text", "")).strip()
        if not text:
            continue
        start = format_timestamp(segment.get("start"))
        end = format_timestamp(segment.get("end"))
        speaker = segment.get("speaker") or "UNKNOWN_SPEAKER"
        lines.append(f"[{start}–{end}] {speaker}: {text}")
    return "\n\n".join(lines)


def segments_dataframe(result, source_file):
    rows = []
    for segment_id, segment in enumerate(result.get("segments", [])):
        text = str(segment.get("text", "")).strip()
        words = segment.get("words", []) or []
        start = segment.get("start")
        end = segment.get("end")
        rows.append({
            "source_file": str(source_file),
            "segment_id": segment_id,
            "start": start,
            "end": end,
            "duration_seconds": (
                float(end) - float(start)
                if start is not None and end is not None else None
            ),
            "speaker": segment.get("speaker"),
            "text": text,
            "n_text_words": len(text.split()),
            "n_aligned_words": len(words),
            "n_words_with_speaker": sum(
                1 for word in words if word.get("speaker") is not None
            ),
        })
    return pd.DataFrame(rows)


def words_dataframe(result, source_file):
    rows = []
    for segment_id, segment in enumerate(result.get("segments", [])):
        segment_speaker = segment.get("speaker")
        for word_id, word in enumerate(segment.get("words", []) or []):
            rows.append({
                "source_file": str(source_file),
                "segment_id": segment_id,
                "word_id": word_id,
                "word": word.get("word"),
                "start": word.get("start"),
                "end": word.get("end"),
                "score": word.get("score"),
                "speaker": word.get("speaker"),
                "segment_speaker": segment_speaker,
            })
    return pd.DataFrame(rows)


def output_paths(audio_path):
    stem = Path(audio_path).stem
    return {
        "asr_json": OUTPUT_DIR / f"{stem}_asr.json",
        "aligned_json": OUTPUT_DIR / f"{stem}_aligned.json",
        "diarization_csv": OUTPUT_DIR / f"{stem}_diarization.csv",
        "final_json": OUTPUT_DIR / f"{stem}_whisperx.json",
        "plain_txt": OUTPUT_DIR / f"{stem}_transcript.txt",
        "speaker_txt": OUTPUT_DIR / f"{stem}_speakers.txt",
        "segments_csv": OUTPUT_DIR / f"{stem}_segments.csv",
        "words_csv": OUTPUT_DIR / f"{stem}_words.csv",
        "metadata_json": OUTPUT_DIR / f"{stem}_run_metadata.json",
    }

## 7. Load the models once

The ASR, English alignment, and diarization models are loaded once and reused across both hearings.

The pipeline follows the same conceptual order used by WhisperX itself: transcription, forced alignment, diarization, then assignment of speaker labels to aligned transcript words and segments.

In [6]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

In [7]:
import time
t0 = time.time()
from huggingface_hub import HfApi
HfApi().whoami(token=HF_TOKEN)
print(f"HF auth check: {time.time() - t0:.1f}s")

HF auth check: 0.1s


In [8]:
import time

t0 = time.time()
print("Loading WhisperX ASR model...")
asr_model = whisperx.load_model(MODEL_NAME, DEVICE, compute_type=COMPUTE_TYPE, language=LANGUAGE)
print(f"ASR model loaded. ({time.time() - t0:.1f}s)")

t1 = time.time()
print("Loading alignment model...")
align_model, align_metadata = whisperx.load_align_model(language_code=LANGUAGE, device=DEVICE)
print(f"Alignment model loaded. ({time.time() - t1:.1f}s)")

diarize_model = None
if ENABLE_DIARIZATION:
    t2 = time.time()
    print("Loading diarization model...")
    diarize_model = DiarizationPipeline(model_name=DIARIZATION_MODEL, token=HF_TOKEN, device=DEVICE)
    print(f"Diarization model loaded. ({time.time() - t2:.1f}s)")
else:
    print("Diarization disabled.")

Loading WhisperX ASR model...


model.bin:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

2026-08-19 23:22:39 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../../../../../opt/homebrew/lib/python3.10/site-packages/whisperx/assets/pytorch_model.bin`


ASR model loaded. (85.8s)
Loading alignment model...
Alignment model loaded. (0.4s)
Loading diarization model...
2026-08-19 23:22:39 - whisperx.diarize - INFO - Loading diarization model: pyannote/speaker-diarization-community-1
Diarization model loaded. (0.4s)


In [ ]:
MODEL_NAME = "small"
asr_model = whisperx.load_model(MODEL_NAME, DEVICE, compute_type=COMPUTE_TYPE, language=LANGUAGE)

In [9]:
print("Loading WhisperX ASR model...")
asr_model = whisperx.load_model(
    MODEL_NAME,
    DEVICE,
    compute_type=COMPUTE_TYPE,
    language=LANGUAGE,
)
print("ASR model loaded.")

print("Loading alignment model...")
align_model, align_metadata = whisperx.load_align_model(
    language_code=LANGUAGE,
    device=DEVICE,
)
print("Alignment model loaded.")

diarize_model = None
if ENABLE_DIARIZATION:
    print("Loading diarization model...")
    diarize_model = DiarizationPipeline(
        model_name=DIARIZATION_MODEL,
        token=HF_TOKEN,
        device=DEVICE,
    )
    print("Diarization model loaded.")
else:
    print("Diarization disabled.")

Loading WhisperX ASR model...
2026-08-19 23:23:03 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../../../../../opt/homebrew/lib/python3.10/site-packages/whisperx/assets/pytorch_model.bin`


ASR model loaded.
Loading alignment model...
Alignment model loaded.
Loading diarization model...
2026-08-19 23:23:04 - whisperx.diarize - INFO - Loading diarization model: pyannote/speaker-diarization-community-1
Diarization model loaded.


## 8. Complete transcription function

`transcribe_one()` is the core of this notebook. For one MP3 it:

1. hashes the source file;
2. loads audio;
3. runs WhisperX ASR and saves `*_asr.json`;
4. performs forced alignment and saves `*_aligned.json` **before** speaker assignment;
5. runs diarization and saves the raw speaker intervals;
6. assigns diarized speakers to the aligned transcript;
7. exports final JSON, plain text, speaker text, segment CSV, word CSV, and run metadata.

Saving each stage before the next transformation makes the process auditable. If a final output already exists and `OVERWRITE=False`, that MP3 is skipped rather than silently replaced.

In [10]:
def transcribe_one(audio_path, overwrite=OVERWRITE):
    """Fully transcribe, align, diarize, and export one recording."""
    audio_path = Path(audio_path)
    paths = output_paths(audio_path)

    if not audio_path.exists():
        raise FileNotFoundError(audio_path)

    if paths["final_json"].exists() and not overwrite:
        return {
            "source_file": str(audio_path),
            "status": "skipped_existing",
            "final_json": str(paths["final_json"]),
        }

    run_started = utc_now_iso()
    source_hash = sha256_file(audio_path)

    print("\n" + "=" * 88)
    print(f"PROCESSING: {audio_path}")
    print("=" * 88)

    # 1. LOAD AUDIO
    print("1/6  Loading audio...")
    audio = whisperx.load_audio(str(audio_path))

    # 2. ASR
    print("2/6  Transcribing with WhisperX ASR...")
    asr_result = asr_model.transcribe(audio, batch_size=BATCH_SIZE)
    asr_payload = {
        "source_file": str(audio_path),
        "language": asr_result.get("language", LANGUAGE),
        "model_name": MODEL_NAME,
        "device": DEVICE,
        "compute_type": COMPUTE_TYPE,
        "batch_size": BATCH_SIZE,
        "segments": asr_result.get("segments", []),
    }
    write_json(asr_payload, paths["asr_json"])

    # 3. FORCED ALIGNMENT
    print("3/6  Aligning transcript to audio...")
    aligned_result = whisperx.align(
        asr_result["segments"],
        align_model,
        align_metadata,
        audio,
        DEVICE,
        return_char_alignments=False,
    )
    aligned_result["language"] = asr_result.get("language", LANGUAGE)
    write_json(aligned_result, paths["aligned_json"])

    # 4. DIARIZATION
    diarization_df = None
    final_result = aligned_result
    if ENABLE_DIARIZATION:
        print("4/6  Diarizing speakers...")
        diarization_df = diarize_model(
            str(audio_path),
            min_speakers=MIN_SPEAKERS,
            max_speakers=MAX_SPEAKERS,
        )
        diarization_df.reset_index(drop=True).to_csv(
            paths["diarization_csv"], index=False
        )

        # 5. SPEAKER ASSIGNMENT
        print("5/6  Assigning speaker labels to aligned transcript...")
        final_result = whisperx.assign_word_speakers(
            diarization_df,
            aligned_result,
        )
    else:
        print("4/6  Diarization disabled.")
        print("5/6  Speaker assignment skipped.")

    final_result["language"] = asr_result.get("language", LANGUAGE)

    # 6. EXPORTS
    print("6/6  Saving final outputs...")
    write_json(final_result, paths["final_json"])
    paths["plain_txt"].write_text(
        make_plain_transcript(final_result), encoding="utf-8"
    )
    if ENABLE_DIARIZATION:
        paths["speaker_txt"].write_text(
            make_speaker_transcript(final_result), encoding="utf-8"
        )

    segment_df = segments_dataframe(final_result, audio_path)
    word_df = words_dataframe(final_result, audio_path)
    segment_df.to_csv(paths["segments_csv"], index=False)
    word_df.to_csv(paths["words_csv"], index=False)

    n_speakers = (
        int(segment_df["speaker"].dropna().nunique())
        if "speaker" in segment_df.columns and not segment_df.empty else 0
    )
    n_segments_without_speaker = (
        int(segment_df["speaker"].isna().sum())
        if "speaker" in segment_df.columns and not segment_df.empty
        else len(segment_df)
    )
    n_words_without_speaker = (
        int(word_df["speaker"].isna().sum())
        if "speaker" in word_df.columns and not word_df.empty
        else len(word_df)
    )

    metadata = {
        "run_started_utc": run_started,
        "run_finished_utc": utc_now_iso(),
        "source_file": str(audio_path),
        "source_sha256": source_hash,
        "source_size_bytes": audio_path.stat().st_size,
        "pipeline": [
            "whisperx_asr",
            "forced_alignment",
            *( ["speaker_diarization", "speaker_assignment"] if ENABLE_DIARIZATION else [] ),
            "exports",
        ],
        "language": LANGUAGE,
        "asr_model": MODEL_NAME,
        "device": DEVICE,
        "compute_type": COMPUTE_TYPE,
        "batch_size": BATCH_SIZE,
        "diarization_enabled": ENABLE_DIARIZATION,
        "diarization_model": DIARIZATION_MODEL if ENABLE_DIARIZATION else None,
        "min_speakers": MIN_SPEAKERS,
        "max_speakers": MAX_SPEAKERS,
        "python_version": sys.version,
        "platform": platform.platform(),
        "torch_version": torch.__version__,
        "whisperx_version": getattr(
            whisperx, "__version__", "version attribute unavailable"
        ),
        "n_segments": int(len(segment_df)),
        "n_aligned_words": int(len(word_df)),
        "n_speaker_clusters": n_speakers,
        "n_segments_without_speaker": n_segments_without_speaker,
        "n_words_without_speaker": n_words_without_speaker,
        "outputs": {k: str(v) for k, v in paths.items()},
    }
    write_json(metadata, paths["metadata_json"])

    summary = {
        "source_file": str(audio_path),
        "status": "completed",
        "source_sha256": source_hash,
        "n_segments": int(len(segment_df)),
        "n_aligned_words": int(len(word_df)),
        "n_speaker_clusters": n_speakers,
        "n_segments_without_speaker": n_segments_without_speaker,
        "n_words_without_speaker": n_words_without_speaker,
        "final_json": str(paths["final_json"]),
        "segments_csv": str(paths["segments_csv"]),
        "words_csv": str(paths["words_csv"]),
        "speaker_transcript": str(paths["speaker_txt"]) if ENABLE_DIARIZATION else None,
        "metadata_json": str(paths["metadata_json"]),
    }

    print("Completed.")
    print(f"  Segments:         {summary['n_segments']}")
    print(f"  Aligned words:    {summary['n_aligned_words']}")
    print(f"  Speaker clusters: {summary['n_speaker_clusters']}")
    print(f"  Final JSON:       {summary['final_json']}")

    del audio, asr_result, asr_payload, aligned_result, final_result
    if diarization_df is not None:
        del diarization_df
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    return summary

## 9. Run the pipeline on every configured MP3

Each recording is isolated in its own `try/except` block. A failure in one hearing is recorded in the manifest and does not erase a successful result from the other hearing.

In [11]:
run_rows = []

for audio_path in AUDIO_FILES:
    try:
        run_rows.append(transcribe_one(audio_path, overwrite=OVERWRITE))
    except Exception as exc:
        print("\nERROR processing:", audio_path)
        print(repr(exc))
        run_rows.append({
            "source_file": str(audio_path),
            "status": "failed",
            "error": repr(exc),
        })
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

run_manifest = pd.DataFrame(run_rows)
manifest_path = OUTPUT_DIR / "run_manifest.csv"
run_manifest.to_csv(manifest_path, index=False)

print("\n" + "=" * 88)
print("RUN COMPLETE")
print("=" * 88)
print("Manifest:", manifest_path.resolve())
display(run_manifest)


PROCESSING: CyrusOne/Cyrusone_Hearing_12032025.mp3
1/6  Loading audio...
2/6  Transcribing with WhisperX ASR...
3/6  Aligning transcript to audio...
4/6  Diarizing speakers...


/opt/homebrew/lib/python3.10/site-packages/pyannote/audio/models/blocks/pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/ReduceOps.cpp:1839.)
  std = sequences.std(dim=-1, correction=1)
/opt/homebrew/lib/python3.10/site-packages/pyannote/audio/pipelines/clustering.py:620: RuntimeWarning: divide by zero encountered in matmul
  centroids = W.T @ train_embeddings.reshape(-1, dimension) / W.sum(0, keepdims=True).T
/opt/homebrew/lib/python3.10/site-packages/pyannote/audio/pipelines/clustering.py:620: RuntimeWarning: overflow encountered in matmul
  centroids = W.T @ train_embeddings.reshape(-1, dimension) / W.sum(0, keepdims=True).T
/opt/homebrew/lib/python3.10/site-packages/pyannote/audio/pipelines/clustering.py:620: RuntimeWarning: invalid value encountered in matmul
  centroids = W.T @ t

5/6  Assigning speaker labels to aligned transcript...
6/6  Saving final outputs...
Completed.
  Segments:         3350
  Aligned words:    42494
  Speaker clusters: 48
  Final JSON:       whisperx_outputs_diarized/Cyrusone_Hearing_12032025_whisperx.json

PROCESSING: CyrusOne/COB_20260323_Imported.mp3
1/6  Loading audio...
2/6  Transcribing with WhisperX ASR...
3/6  Aligning transcript to audio...
4/6  Diarizing speakers...


/opt/homebrew/lib/python3.10/site-packages/pyannote/audio/pipelines/clustering.py:620: RuntimeWarning: divide by zero encountered in matmul
  centroids = W.T @ train_embeddings.reshape(-1, dimension) / W.sum(0, keepdims=True).T
/opt/homebrew/lib/python3.10/site-packages/pyannote/audio/pipelines/clustering.py:620: RuntimeWarning: overflow encountered in matmul
  centroids = W.T @ train_embeddings.reshape(-1, dimension) / W.sum(0, keepdims=True).T
/opt/homebrew/lib/python3.10/site-packages/pyannote/audio/pipelines/clustering.py:620: RuntimeWarning: invalid value encountered in matmul
  centroids = W.T @ train_embeddings.reshape(-1, dimension) / W.sum(0, keepdims=True).T


5/6  Assigning speaker labels to aligned transcript...
6/6  Saving final outputs...
Completed.
  Segments:         2888
  Aligned words:    33486
  Speaker clusters: 79
  Final JSON:       whisperx_outputs_diarized/COB_20260323_Imported_whisperx.json

RUN COMPLETE
Manifest: /Users/yams/Library/CloudStorage/Dropbox/AEI_magazine_datacenters/whisperx_outputs_diarized/run_manifest.csv


,source_file,status,source_sha256,n_segments,n_aligned_words,n_speaker_clusters,n_segments_without_speaker,n_words_without_speaker,final_json,segments_csv,words_csv,speaker_transcript,metadata_json
0,CyrusOne/Cyrusone_Hearing_12032025.mp3,completed,dfc4dfef932f09c8122ea820d9374a4e3d58ecd4e314a3...,3350,42494,48,4,25,whisperx_outputs_diarized/Cyrusone_Hearing_120...,whisperx_outputs_diarized/Cyrusone_Hearing_120...,whisperx_outputs_diarized/Cyrusone_Hearing_120...,whisperx_outputs_diarized/Cyrusone_Hearing_120...,whisperx_outputs_diarized/Cyrusone_Hearing_120...
1,CyrusOne/COB_20260323_Imported.mp3,completed,dde34af32fe56fc892c33d814c0c48add2d25262c29a26...,2888,33486,79,2,5,whisperx_outputs_diarized/COB_20260323_Importe...,whisperx_outputs_diarized/COB_20260323_Importe...,whisperx_outputs_diarized/COB_20260323_Importe...,whisperx_outputs_diarized/COB_20260323_Importe...,whisperx_outputs_diarized/COB_20260323_Importe...


## 10. QA: speaker-assignment coverage

Diarization is probabilistic. This cell reports whether speaker labels were attached broadly enough to support manual inspection; it does **not** prove that the speaker identities or boundaries are correct.

It reports missing speaker assignments at both segment and word level for each hearing.

In [12]:
qa_rows = []

for audio_path in AUDIO_FILES:
    paths = output_paths(audio_path)
    if not paths["segments_csv"].exists():
        continue

    segments = pd.read_csv(paths["segments_csv"])
    words = pd.read_csv(paths["words_csv"]) if paths["words_csv"].exists() else pd.DataFrame()

    n_segments = len(segments)
    n_words = len(words)
    segment_missing = (
        int(segments["speaker"].isna().sum())
        if "speaker" in segments.columns else n_segments
    )
    word_missing = (
        int(words["speaker"].isna().sum())
        if "speaker" in words.columns else n_words
    )

    qa_rows.append({
        "source_file": str(audio_path),
        "n_segments": n_segments,
        "n_speaker_clusters": (
            int(segments["speaker"].dropna().nunique())
            if "speaker" in segments.columns else 0
        ),
        "segments_without_speaker": segment_missing,
        "share_segments_without_speaker": segment_missing / n_segments if n_segments else None,
        "n_aligned_words": n_words,
        "words_without_speaker": word_missing,
        "share_words_without_speaker": word_missing / n_words if n_words else None,
    })

qa_df = pd.DataFrame(qa_rows)
if qa_df.empty:
    print("No completed segment outputs were found.")
else:
    display(qa_df)
    qa_path = OUTPUT_DIR / "diarization_qa.csv"
    qa_df.to_csv(qa_path, index=False)
    print("Saved:", qa_path.resolve())

,source_file,n_segments,n_speaker_clusters,segments_without_speaker,share_segments_without_speaker,n_aligned_words,words_without_speaker,share_words_without_speaker
0,CyrusOne/Cyrusone_Hearing_12032025.mp3,3350,48,4,0.001194,42494,25,0.000588
1,CyrusOne/COB_20260323_Imported.mp3,2888,79,2,0.000693,33486,5,0.000149


Saved: /Users/yams/Library/CloudStorage/Dropbox/AEI_magazine_datacenters/whisperx_outputs_diarized/diarization_qa.csv


## 11. Preview the diarized transcripts

Show the first 25 aligned segments from each hearing as a sanity check. This is not a substitute for listening to the audio around important quotations or speaker changes.

In [13]:
for audio_path in AUDIO_FILES:
    paths = output_paths(audio_path)
    if not paths["segments_csv"].exists():
        continue

    print("\n" + "=" * 88)
    print(audio_path.name)
    print("=" * 88)

    preview = pd.read_csv(paths["segments_csv"])
    display(preview[["segment_id", "start", "end", "speaker", "text"]].head(25))


Cyrusone_Hearing_12032025.mp3


,segment_id,start,end,speaker,text
0,0,0.740,1.941,SPEAKER_12,THE BOARD WILL COME TO ORDER.
1,1,3.301,5.102,SPEAKER_12,INVOCATION BY MS. FULGENZI.
2,2,5.982,7.662,SPEAKER_12,PLEDGE OF ALLEGIANCE BY MR. CHUMMELIN.
3,3,7.682,15.565,SPEAKER_16,"DEVON, FOLLOW ME OUT."
4,4,15.605,17.245,SPEAKER_41,PLEASE GUIDE US.
5,5,17.746,23.287,SPEAKER_41,PLEASE HELP US TO SHOW OUR NEIGHBORS COMPASSIO...
6,6,24.028,28.589,SPEAKER_41,PLEASE HELP US ALL TO LISTEN AND TO LEARN TONI...
7,7,30.087,35.011,SPEAKER_41,"And dear Lord, please bless everyone in this r..."
8,8,35.071,39.274,SPEAKER_41,"May everyone enjoy faith, family, and this sea..."
9,9,39.754,39.934,SPEAKER_41,Amen.



COB_20260323_Imported.mp3


,segment_id,start,end,speaker,text
0,0,0.945,11.469,SPEAKER_76,We ask of you this evening to help us in our d...
1,1,12.169,16.310,SPEAKER_76,We know that you will guide us for the betterm...
2,2,16.751,20.232,SPEAKER_76,"We humbly now ask and do so in your name, Jesus."
3,3,20.252,20.352,SPEAKER_76,Amen.
4,4,20.372,29.195,SPEAKER_11,"Mr. Chairman, before each group of us talking ..."
5,5,29.619,32.100,SPEAKER_68,but our track and field did better than what w...
6,6,32.440,34.902,SPEAKER_68,"So they are in meets tonight, so they won't be..."
7,7,35.242,36.402,SPEAKER_68,So I'm going to substitute.
8,8,36.422,37.583,SPEAKER_28,So you have to give the pledge.
9,9,37.823,39.664,SPEAKER_68,"Yeah, apparently I'm the slowest runner."


## 12. Create a speaker-identification crosswalk template

Diarization produces anonymous voice clusters. This cell creates a blank annotation template for later mapping each `(source_file, speaker)` pair to a real person and role.

Fill `speaker_name`, `speaker_role`, `identity_evidence`, and `notes` manually after checking audio, agendas, self-introductions, or other source material. The transcription pipeline itself does not guess identities.

In [14]:
crosswalk_rows = []

for audio_path in AUDIO_FILES:
    paths = output_paths(audio_path)
    if not paths["segments_csv"].exists():
        continue

    segments = pd.read_csv(paths["segments_csv"])
    if "speaker" not in segments.columns:
        continue

    for speaker in sorted(segments["speaker"].dropna().astype(str).unique()):
        speaker_segments = segments[segments["speaker"].astype(str) == speaker]
        crosswalk_rows.append({
            "source_file": str(audio_path),
            "speaker": speaker,
            "n_segments": len(speaker_segments),
            "speaker_name": "",
            "speaker_role": "",
            "identity_evidence": "",
            "notes": "",
        })

crosswalk = pd.DataFrame(crosswalk_rows)
crosswalk_path = OUTPUT_DIR / "speaker_crosswalk_template.csv"
crosswalk.to_csv(crosswalk_path, index=False)
display(crosswalk)
print("Saved:", crosswalk_path.resolve())

,source_file,speaker,n_segments,speaker_name,speaker_role,identity_evidence,notes
0,CyrusOne/Cyrusone_Hearing_12032025.mp3,SPEAKER_00,8,,,,
1,CyrusOne/Cyrusone_Hearing_12032025.mp3,SPEAKER_01,31,,,,
2,CyrusOne/Cyrusone_Hearing_12032025.mp3,SPEAKER_02,23,,,,
3,CyrusOne/Cyrusone_Hearing_12032025.mp3,SPEAKER_03,4,,,,
4,CyrusOne/Cyrusone_Hearing_12032025.mp3,SPEAKER_04,12,,,,
...,...,...,...,...,...,...,...
122,CyrusOne/COB_20260323_Imported.mp3,SPEAKER_74,15,,,,
123,CyrusOne/COB_20260323_Imported.mp3,SPEAKER_75,19,,,,
124,CyrusOne/COB_20260323_Imported.mp3,SPEAKER_76,21,,,,
125,CyrusOne/COB_20260323_Imported.mp3,SPEAKER_77,7,,,,


Saved: /Users/yams/Library/CloudStorage/Dropbox/AEI_magazine_datacenters/whisperx_outputs_diarized/speaker_crosswalk_template.csv


## 13. Files to use downstream

After auditing diarization and filling the speaker crosswalk:

- use `*_transcript.txt` when speaker identity is irrelevant;
- use `*_segments.csv` for timestamped text and speaker-turn reconstruction;
- use `*_words.csv` for fine-grained word timing and speaker inspection;
- use `*_speakers.txt` for manual reading and listening checks;
- use `*_whisperx.json` when the fullest aligned representation is needed;
- retain `*_asr.json`, `*_aligned.json`, `*_diarization.csv`, and `*_run_metadata.json` as the provenance trail.

The downstream NLP notebook should consume these outputs. It should not rerun WhisperX.